# GLB File Analysis for Three.js Compatibility

This notebook analyzes the generated GLB file to ensure it's properly set up for three.js and other 3D visualization libraries. We'll check:

1. **Coordinate system and centering** - Ensure model is properly positioned
2. **Geometry properties** - Verify mesh structure and normals
3. **Material compatibility** - Check material definitions
4. **Bounding box** - Analyze size and positioning
5. **Three.js best practices** - Apply any necessary corrections

In [1]:
# Import Required Libraries
import numpy as np
import trimesh
from pathlib import Path
import json
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Try to import pygltflib for detailed GLB analysis
try:
    import pygltflib
    HAS_PYGLTFLIB = True
    print("✓ pygltflib available for detailed GLB analysis")
except ImportError:
    HAS_PYGLTFLIB = False
    print("⚠ pygltflib not available - will use trimesh for basic analysis")
    print("  Install with: pip install pygltflib")

print("Libraries loaded successfully!")

⚠ pygltflib not available - will use trimesh for basic analysis
  Install with: pip install pygltflib
Libraries loaded successfully!


In [3]:
# Load and Inspect GLB File
GLB_PATH = Path(r"C:\CodingProjects\bioctree\external\out\lh_pial_fsaverage.glb")

if not GLB_PATH.exists():
    raise FileNotFoundError(f"GLB file not found: {GLB_PATH}")

print(f"Analyzing GLB file: {GLB_PATH}")
print(f"File size: {GLB_PATH.stat().st_size / 1024 / 1024:.2f} MB")

# Load with trimesh
scene = trimesh.load(GLB_PATH)
print(f"Loaded as: {type(scene)}")

# Check if it's a single mesh or scene and extract geometry
if hasattr(scene, 'vertices'):
    print("Single mesh detected")
    vertices = scene.vertices
    faces = scene.faces
    has_normals = hasattr(scene, 'vertex_normals') and scene.vertex_normals is not None
    mesh = scene
elif hasattr(scene, 'geometry') and len(scene.geometry) > 0:
    print("Scene with multiple meshes detected")
    # Get the first/main mesh from the scene
    mesh_name = list(scene.geometry.keys())[0]
    mesh = scene.geometry[mesh_name]
    vertices = mesh.vertices
    faces = mesh.faces
    has_normals = hasattr(mesh, 'vertex_normals') and mesh.vertex_normals is not None
    print(f"Using mesh: {mesh_name}")
else:
    # Try to dump the scene to a combined mesh
    print("Scene detected - converting to combined mesh")
    mesh = scene.dump(concatenate=True)
    vertices = mesh.vertices
    faces = mesh.faces
    has_normals = hasattr(mesh, 'vertex_normals') and mesh.vertex_normals is not None

print(f"Vertices: {len(vertices):,}")
print(f"Faces: {len(faces):,}")
print(f"Has vertex normals: {has_normals}")

Analyzing GLB file: C:\CodingProjects\bioctree\external\out\lh_pial_fsaverage.glb
File size: 7.50 MB
Loaded as: <class 'trimesh.scene.scene.Scene'>
Scene with multiple meshes detected
Using mesh: geometry_0
Vertices: 163,842
Faces: 327,680
Has vertex normals: True


In [4]:
# Check Model Geometry and Bounds
print("=== GEOMETRY ANALYSIS ===")

# Bounding box analysis
bbox_min = vertices.min(axis=0)
bbox_max = vertices.max(axis=0)
bbox_center = (bbox_min + bbox_max) / 2
bbox_size = bbox_max - bbox_min

print(f"Bounding box:")
print(f"  Min: [{bbox_min[0]:.2f}, {bbox_min[1]:.2f}, {bbox_min[2]:.2f}]")
print(f"  Max: [{bbox_max[0]:.2f}, {bbox_max[1]:.2f}, {bbox_max[2]:.2f}]")
print(f"  Center: [{bbox_center[0]:.2f}, {bbox_center[1]:.2f}, {bbox_center[2]:.2f}]")
print(f"  Size: [{bbox_size[0]:.2f}, {bbox_size[1]:.2f}, {bbox_size[2]:.2f}]")

# Check if model is centered at origin
distance_from_origin = np.linalg.norm(bbox_center)
print(f"Distance from origin: {distance_from_origin:.2f}")

# Determine if centering is needed
CENTERING_THRESHOLD = 1.0  # If center is more than 1 unit from origin
needs_centering = distance_from_origin > CENTERING_THRESHOLD

print(f"Needs centering: {'YES' if needs_centering else 'NO'}")

# Check coordinate system (Brain coordinates are typically in mm)
print(f"\nCoordinate system analysis:")
print(f"  Coordinate range suggests: {'millimeters (brain data)' if bbox_size.max() > 100 else 'normalized units'}")

# Mesh quality checks
if hasattr(mesh, 'is_watertight'):
    print(f"  Watertight: {mesh.is_watertight}")
if hasattr(mesh, 'is_winding_consistent'):
    print(f"  Consistent winding: {mesh.is_winding_consistent}")
if hasattr(mesh, 'euler_number'):
    print(f"  Euler number: {mesh.euler_number}")

=== GEOMETRY ANALYSIS ===
Bounding box:
  Min: [0.00, 0.00, 0.00]
  Max: [70.58, 173.79, 125.08]
  Center: [35.29, 86.89, 62.54]
  Size: [70.58, 173.79, 125.08]
Distance from origin: 112.73
Needs centering: YES

Coordinate system analysis:
  Coordinate range suggests: millimeters (brain data)
  Watertight: True
  Consistent winding: True
  Euler number: 2


In [5]:
# Analyze Model Centering for Three.js
print("=== THREE.JS COMPATIBILITY ANALYSIS ===")

# Three.js best practices for brain models:
# 1. Model should be centered at origin for proper camera controls
# 2. Reasonable scale (not too large/small)
# 3. Proper orientation (typically Y-up)

# Check scale appropriateness
scale_factor = 1.0
max_dimension = bbox_size.max()

if max_dimension > 1000:  # Brain in mm, too large for typical three.js scenes
    scale_factor = 100.0 / max_dimension  # Scale to ~100 units
    print(f"Model is large ({max_dimension:.1f}mm) - recommend scaling by {scale_factor:.4f}")
elif max_dimension < 1:  # Too small
    scale_factor = 100.0 / max_dimension  # Scale up
    print(f"Model is small ({max_dimension:.4f}) - recommend scaling by {scale_factor:.2f}")
else:
    print(f"Model scale ({max_dimension:.1f}) is appropriate for three.js")

# Check orientation (brain models are often in neurological coordinates)
print(f"\nOrientation analysis:")
print(f"  X-axis span: {bbox_size[0]:.1f} (left-right)")
print(f"  Y-axis span: {bbox_size[1]:.1f} (anterior-posterior)")  
print(f"  Z-axis span: {bbox_size[2]:.1f} (inferior-superior)")

# Recommend transformations
print(f"\nRecommended transformations for three.js:")
if needs_centering:
    print(f"  1. Center at origin: translate by [{-bbox_center[0]:.2f}, {-bbox_center[1]:.2f}, {-bbox_center[2]:.2f}]")
if scale_factor != 1.0:
    print(f"  2. Scale uniformly by {scale_factor:.4f}")
    
# Check if any transformation is needed
needs_transformation = needs_centering or (scale_factor != 1.0)
print(f"\nNeeds transformation: {'YES' if needs_transformation else 'NO'}")

=== THREE.JS COMPATIBILITY ANALYSIS ===
Model scale (173.8) is appropriate for three.js

Orientation analysis:
  X-axis span: 70.6 (left-right)
  Y-axis span: 173.8 (anterior-posterior)
  Z-axis span: 125.1 (inferior-superior)

Recommended transformations for three.js:
  1. Center at origin: translate by [-35.29, -86.89, -62.54]

Needs transformation: YES


In [6]:
# Validate Material Properties and Normals
print("=== MATERIAL AND NORMALS ANALYSIS ===")

# Check vertex normals
if hasattr(mesh, 'vertex_normals') and mesh.vertex_normals is not None:
    normals = mesh.vertex_normals
    print(f"Vertex normals: Present ({len(normals):,} normals)")
    
    # Check normal vectors quality
    normal_lengths = np.linalg.norm(normals, axis=1)
    normalized_properly = np.allclose(normal_lengths, 1.0, atol=1e-3)
    print(f"  Normalized properly: {normalized_properly}")
    print(f"  Length range: [{normal_lengths.min():.3f}, {normal_lengths.max():.3f}]")
else:
    print("Vertex normals: MISSING - will affect lighting in three.js")

# Check materials
if hasattr(mesh, 'visual') and mesh.visual is not None:
    print(f"Visual properties: Present")
    if hasattr(mesh.visual, 'material'):
        print(f"  Material type: {type(mesh.visual.material)}")
    if hasattr(mesh.visual, 'vertex_colors') and mesh.visual.vertex_colors is not None:
        print(f"  Vertex colors: Present ({len(mesh.visual.vertex_colors):,} colors)")
else:
    print("Visual properties: Basic (will use default three.js material)")

# Check mesh topology
print(f"\nMesh topology:")
print(f"  Face type: {'Triangular' if faces.shape[1] == 3 else 'Mixed/Other'}")

# UV coordinates (for texture mapping)
if hasattr(mesh, 'visual') and hasattr(mesh.visual, 'uv'):
    print(f"  UV coordinates: Present")
else:
    print("  UV coordinates: Not present (no texture mapping)")

=== MATERIAL AND NORMALS ANALYSIS ===
Vertex normals: Present (163,842 normals)
  Normalized properly: True
  Length range: [1.000, 1.000]
Visual properties: Present
  Vertex colors: Present (163,842 colors)

Mesh topology:
  Face type: Triangular
  UV coordinates: Not present (no texture mapping)


In [ ]:
# Visualize Model Bounds and Structure
print("=== VISUALIZATION ===")

# Create 3D visualization of the model bounds and structure
fig = plt.figure(figsize=(15, 5))

# Plot 1: Bounding box
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(*bbox_center, color='red', s=100, label='Center')
ax1.plot([bbox_min[0], bbox_max[0]], [bbox_center[1], bbox_center[1]], [bbox_center[2], bbox_center[2]], 'b-', alpha=0.6)
ax1.plot([bbox_center[0], bbox_center[0]], [bbox_min[1], bbox_max[1]], [bbox_center[2], bbox_center[2]], 'g-', alpha=0.6)
ax1.plot([bbox_center[0], bbox_center[0]], [bbox_center[1], bbox_center[1]], [bbox_min[2], bbox_max[2]], 'r-', alpha=0.6)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_title('Model Bounding Box')
ax1.legend()

# Plot 2: Vertex distribution (sample)
ax2 = fig.add_subplot(132, projection='3d')
sample_indices = np.random.choice(len(vertices), min(1000, len(vertices)), replace=False)
sample_verts = vertices[sample_indices]
ax2.scatter(sample_verts[:, 0], sample_verts[:, 1], sample_verts[:, 2], 
           alpha=0.3, s=1, c=sample_verts[:, 2], cmap='viridis')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.set_title('Vertex Distribution (Sample)')

# Plot 3: Coordinate distribution histograms
ax3 = fig.add_subplot(133)
ax3.hist(vertices[:, 0], bins=50, alpha=0.5, label='X', density=True)
ax3.hist(vertices[:, 1], bins=50, alpha=0.5, label='Y', density=True)
ax3.hist(vertices[:, 2], bins=50, alpha=0.5, label='Z', density=True)
ax3.axvline(0, color='black', linestyle='--', alpha=0.5, label='Origin')
ax3.set_xlabel('Coordinate Value')
ax3.set_ylabel('Density')
ax3.set_title('Coordinate Distribution')
ax3.legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nCoordinate statistics:")
for i, axis in enumerate(['X', 'Y', 'Z']):
    print(f"  {axis}: mean={vertices[:, i].mean():.2f}, std={vertices[:, i].std():.2f}")

In [7]:
# Export Corrected Model (if needed)
if needs_transformation:
    print("=== CREATING CORRECTED MODEL ===")
    
    # Create a copy of the mesh for transformation
    corrected_mesh = mesh.copy()
    
    # Apply centering transformation
    if needs_centering:
        print(f"Centering model...")
        corrected_mesh.vertices = corrected_mesh.vertices - bbox_center
        print(f"  Translated by: [{-bbox_center[0]:.2f}, {-bbox_center[1]:.2f}, {-bbox_center[2]:.2f}]")
    
    # Apply scaling transformation
    if scale_factor != 1.0:
        print(f"Scaling model...")
        corrected_mesh.vertices = corrected_mesh.vertices * scale_factor
        print(f"  Scaled by: {scale_factor:.4f}")
    
    # Ensure normals are recalculated after transformation
    if hasattr(corrected_mesh, 'vertex_normals'):
        corrected_mesh.vertex_normals = None  # Force recalculation
    
    # Export corrected model
    output_path = GLB_PATH.parent / f"{GLB_PATH.stem}_corrected.glb"
    corrected_mesh.export(str(output_path))
    
    print(f"Corrected model exported to: {output_path}")
    
    # Verify corrected model
    new_bbox_min = corrected_mesh.vertices.min(axis=0)
    new_bbox_max = corrected_mesh.vertices.max(axis=0)
    new_bbox_center = (new_bbox_min + new_bbox_max) / 2
    new_distance_from_origin = np.linalg.norm(new_bbox_center)
    
    print(f"Corrected model stats:")
    print(f"  New center: [{new_bbox_center[0]:.3f}, {new_bbox_center[1]:.3f}, {new_bbox_center[2]:.3f}]")
    print(f"  Distance from origin: {new_distance_from_origin:.3f}")
    print(f"  New size: {(new_bbox_max - new_bbox_min).max():.2f}")
    
else:
    print("=== NO CORRECTIONS NEEDED ===")
    print("The GLB file is already properly set up for three.js!")
    print("You can use it directly in your three.js application.")

=== CREATING CORRECTED MODEL ===
Centering model...
  Translated by: [-35.29, -86.89, -62.54]
Corrected model exported to: C:\CodingProjects\bioctree\external\out\lh_pial_fsaverage_corrected.glb
Corrected model stats:
  New center: [0.000, 0.000, 0.000]
  Distance from origin: 0.000
  New size: 173.79


## Three.js Usage Recommendations

Based on the analysis above, here are the recommendations for using this GLB file in three.js:

### Loading the Model
```javascript
import { GLTFLoader } from 'three/examples/jsm/loaders/GLTFLoader.js';

const loader = new GLTFLoader();
loader.load('path/to/your/model.glb', (gltf) => {
    const model = gltf.scene;
    scene.add(model);
});
```

### Camera Setup
For brain models, consider these camera settings:
- **Position**: Place camera at a reasonable distance based on model size
- **Controls**: Use OrbitControls for interactive viewing
- **Target**: Set camera target to model center (should be origin if corrected)

### Lighting Setup
Brain models typically benefit from:
- **Ambient Light**: Soft overall illumination
- **Directional Light**: Main lighting to show surface details
- **Point Lights**: Optional highlights for specific areas

### Material Considerations
- The model uses basic materials - you may want to enhance with custom shaders
- Consider adding subsurface scattering for more realistic brain appearance
- Vertex colors can be used to display morphometric data (thickness, curvature, etc.)